In [0]:
# Project setup: makes the src/ package importable and loads the config
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

from src.utils.config_loader import load_config
from src.utils.spark_helpers import run_sql_file

cfg = load_config()

In [0]:
run_sql_file(spark, "gold/01_create_gold_games.sql", cfg)

In [0]:
run_sql_file(spark, "gold/02_enable_column_mapping.sql", cfg)

In [0]:
run_sql_file(spark, "gold/03_drop_helper_columns.sql", cfg)

In [0]:
%sql
select * from steam_data.default.gold_games

We will use this table to answer our business problem

Let us start with some visualization

In [0]:
%sql
select genre, count(*) as count from steam_data.default.game_genres group by genre order by count desc

Databricks visualization. Run in Databricks to view.

Let us see how the median playtime forever is distributed

In [0]:
games_df = spark.table("steam_data.default.gold_games")

genres_df=spark.table("steam_data.default.game_genres")

In [0]:
gameJgenres=games_df.join(genres_df,on="app_id",how="inner")

In [0]:
%sql
SELECT
  COUNT(*) AS games,
  ROUND(AVG(median_playtime_forever)/60, 1) AS mean_hours,
  ROUND(PERCENTILE(median_playtime_forever, 0.5)/60, 1) AS median_hours,
  ROUND(PERCENTILE(median_playtime_forever, 0.25)/60, 1) AS p25_hours,
  ROUND(PERCENTILE(median_playtime_forever, 0.75)/60, 1) AS p75_hours,
  ROUND(PERCENTILE(median_playtime_forever, 0.95)/60, 1) AS p95_hours
FROM steam_data.default.gold_games
WHERE median_playtime_forever > 0

As we can see above , the mean is very high compared to the median , confirming a right skew . This means few games pull the mean up while the median stays behind. Hence median is a better metric to evaluate the engagement. 

In [0]:
import matplotlib.pyplot as plt
import numpy as np
from pyspark.sql import functions as F

pdf = (gameJgenres
       .filter("median_playtime_forever > 0")
       .withColumn("log_playtime", F.log10("median_playtime_forever"))
       .select("genre", "log_playtime")
       .toPandas())

counts = pdf["genre"].value_counts()
genres = counts[counts >= 30].index.tolist()      

cols = 4
rows = int(np.ceil(len(genres) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3 * rows), sharex=True)
axes = np.atleast_1d(axes).flatten()

bins = np.linspace(pdf["log_playtime"].min(), pdf["log_playtime"].max(), 30)

for ax, g in zip(axes, genres):
    vals = pdf.loc[pdf["genre"] == g, "log_playtime"]
    ax.hist(vals, bins=bins)
    ax.axvline(vals.median(), color="red", linestyle="--", linewidth=1)   
    ax.set_title(f"{g} (n={counts[g]})", fontsize=10)

for ax in axes[len(genres):]:      
    ax.axis("off")

fig.supxlabel("log10(median playtime, minutes)")
fig.supylabel("number of games")
plt.tight_layout()
plt.show()

Playtime distributions are similar across genres, all roughly log-normal. Strategy and RPG show slightly higher typical playtime (about 5 hours) than Racing and MMO (about 2.5 to 3 hours), but the differences between genres are small relative to the variation within each genre.

In [0]:
genre_stats = (
    gameJgenres
    .filter("median_playtime_forever > 0")
    .groupBy("genre")
    .agg(
        F.countDistinct("app_id").alias("num_games"),
        F.round(F.expr("percentile(median_playtime_forever, 0.5)") / 60, 1).alias("median_of_medians_hrs"),
        F.round(F.avg("median_playtime_forever") / 60, 1).alias("avg_of_medians_hrs")
    )
    .withColumn("skew_ratio", F.round(F.col("avg_of_medians_hrs") / F.col("median_of_medians_hrs"), 2))
    .filter("num_games >= 30")
    .orderBy(F.desc("median_of_medians_hrs"))
)

display(genre_stats)

Databricks visualization. Run in Databricks to view.


RPG and Strategy have the highest median playtime of the genres analysed, at roughly 5.4 and 5.1 hours per game, compared with about 2.8 hours for Racing and Sports. The gap between genres is modest, though. Playtime varies far more within a genre than between genres, so genre is only a weak indicator of engagement.

**Limitation:** this analysis only covers games with recorded playtime (about 12k of 138k games, roughly 9%). These are likely to be more established titles, so the results describe games that gained traction rather than all games on Steam.

The ranking is plausible, since RPG and Strategy games tend to be longer and more replayable than Racing or Sports games. This is a possible explanation and was not tested in this analysis.

We will do the same for Single Player VS Multiplayer

In [0]:
categories_df=spark.table("steam_data.default.game_categories")
gameJcat=games_df.join(categories_df,on="app_id",how="inner")


In [0]:
display(categories_df)

In [0]:
display(gameJcat)

In [0]:
import matplotlib.pyplot as plt
import numpy as np
from pyspark.sql import functions as F

keep = ["Single-player", "Multi-player"]

pdf = (gameJcat
       .filter("median_playtime_forever > 0")
       .filter(F.col("category").isin(keep))
       .withColumn("log_playtime", F.log10("median_playtime_forever"))
       .select("category", "log_playtime")
       .toPandas())

bins = np.linspace(pdf["log_playtime"].min(), pdf["log_playtime"].max(), 30)

fig, ax = plt.subplots(figsize=(9, 5))
for cat, color in zip(keep, ["tab:blue", "tab:orange"]):
    vals = pdf.loc[pdf["category"] == cat, "log_playtime"]
    ax.hist(vals, bins=bins, alpha=0.5, color=color, density=True,
            label=f"{cat} (n={len(vals)})")
    ax.axvline(vals.median(), color=color, linestyle="--", linewidth=1.5)

ax.set_xlabel("log10(median playtime, minutes)")
ax.set_ylabel("share of games")
ax.legend()
plt.tight_layout()
plt.show()

Single-player and multiplayer games show nearly identical distributions of median playtime, with essentially the same median . Multiplayer games have a slightly heavier upper tail, driven by a small number of very high-engagement titles

Adding a Single_Player and 